## environment setup and library imports
professional setup for data analysis, visualization, and machine learning experiments.

In [17]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
import pickle

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)


## dataset loading and initialization
loading all datasets required for the integrated football analytics pipeline.

In [18]:
games = pd.read_csv(r'd:\Projects\WC26Analytics\games.csv', low_memory=False)
club_games = pd.read_csv(r'd:\Projects\WC26Analytics\club_games.csv')
game_lineups = pd.read_csv(r'd:\Projects\WC26Analytics\game_lineups.csv')
players = pd.read_csv(r'd:\Projects\WC26Analytics\players.csv')
appearances = pd.read_csv(r'd:\Projects\WC26Analytics\appearances.csv')
player_valuations = pd.read_csv(r'd:\Projects\WC26Analytics\player_valuations.csv')
transfers = pd.read_csv(r'd:\Projects\WC26Analytics\transfers.csv')
clubs = pd.read_csv(r'd:\Projects\WC26Analytics\clubs.csv')
competitions = pd.read_csv(r'd:\Projects\WC26Analytics\competitions.csv')
countries = pd.read_csv(r'd:\Projects\WC26Analytics\countries.csv')
national_teams = pd.read_csv(r'd:\Projects\WC26Analytics\national_teams.csv')
game_events = pd.read_csv(r'd:\Projects\WC26Analytics\game_events.csv')

for df in [games, game_lineups, player_valuations, transfers, appearances]:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')

## data preparation
cleaning and preparing the merged dataset for feature engineering and modeling.

In [19]:
match_data = club_games.merge(games[['game_id', 'date', 'season', 'competition_id']],
                               on='game_id', how='left')
match_data = match_data.merge(competitions[['competition_id', 'type']],
                               left_on='competition_id', right_on='competition_id', how='left')

match_data = match_data.dropna(subset=['date', 'is_win', 'own_goals', 'opponent_goals'])
match_data['year'] = match_data['date'].dt.year
# match_data = match_data[match_data['year'] >= 2014].copy().sort_values('date').reset_index(drop=True)
match_data = match_data.copy().sort_values('date').reset_index(drop=True)

## feature engineering
constructing performance, market value, transfer, and event-based features.

In [20]:
df = match_data.copy()

df['goal_diff'] = df['own_goals'] - df['opponent_goals']
df['total_goals'] = df['own_goals'] + df['opponent_goals']
df['is_home'] = (df['hosting'] == 'Home').astype(int)

# rolling statistics for team (20-match window)
for club_id in df['club_id'].unique():
    mask = df['club_id'] == club_id
    club_df = df[mask].sort_values('date').reset_index(drop=True)

    wins = club_df['is_win'].rolling(20, min_periods=1).mean()
    gf = club_df['own_goals'].rolling(20, min_periods=1).mean()
    ga = club_df['opponent_goals'].rolling(20, min_periods=1).mean()

    df.loc[mask, 'team_win_rate'] = wins.values
    df.loc[mask, 'team_gf'] = gf.values
    df.loc[mask, 'team_ga'] = ga.values

# same for opponent
for club_id in df['opponent_id'].unique():
    mask = df['opponent_id'] == club_id
    club_df = df[mask].sort_values('date').reset_index(drop=True)

    wins = club_df['is_win'].rolling(20, min_periods=1).mean()
    gf = club_df['own_goals'].rolling(20, min_periods=1).mean()
    ga = club_df['opponent_goals'].rolling(20, min_periods=1).mean()

    df.loc[mask, 'opp_win_rate'] = wins.values
    df.loc[mask, 'opp_gf'] = gf.values
    df.loc[mask, 'opp_ga'] = ga.values

# derived features
df['strength_diff'] = (df['team_gf'] - df['team_ga']) - (df['opp_gf'] - df['opp_ga'])
df['team_vs_opp'] = df['team_win_rate'] - df['opp_win_rate']

# squad market values (from player_valuations)
squad_vals = player_valuations.groupby('current_club_id')['market_value_in_eur'].agg(['mean', 'sum']).reset_index()
squad_vals.columns = ['club_id', 'avg_val', 'total_val']

df = df.merge(squad_vals, left_on='club_id', right_on='club_id', how='left')

# for opponent, rename columns to avoid collision
squad_vals_opp = squad_vals.copy()
squad_vals_opp.columns = ['opponent_id', 'avg_val_o', 'total_val_o']
df = df.merge(squad_vals_opp, left_on='opponent_id', right_on='opponent_id', how='left')

df['value_diff'] = df['total_val'].fillna(0) - df['total_val_o'].fillna(0)
df['player_qual_diff'] = df['avg_val'].fillna(0) - df['avg_val_o'].fillna(0)

# transfers activity
trans_in = transfers.groupby('to_club_id').size().reset_index(name='n_trans')
trans_dict = dict(zip(trans_in['to_club_id'], trans_in['n_trans']))

df['transfers_team'] = df['club_id'].map(trans_dict).fillna(0)
df['transfers_opp'] = df['opponent_id'].map(trans_dict).fillna(0)

# game events
events_count = game_events.groupby(['game_id', 'club_id']).size().reset_index(name='n_events')
events_dict = {}
for gid, club, count in zip(events_count['game_id'], events_count['club_id'], events_count['n_events']):
    events_dict[(gid, club)] = count

df['events_team'] = df.apply(lambda r: events_dict.get((r['game_id'], r['club_id']), 0), axis=1)
df['events_opp'] = df.apply(lambda r: events_dict.get((r['game_id'], r['opponent_id']), 0), axis=1)

# fill NaN
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

## statistical analysis
analyzing feature correlations and target relationships.

In [21]:
corr_with_target = df[numeric_cols].corr()['is_win'].sort_values(ascending=False)

print(f"\n  Top 15 correlations:")
for feat, corr in corr_with_target.head(15).items():
    if feat != 'is_win':
        print(f"    {feat:35s}: {corr:.4f}")



  Top 15 correlations:
    goal_diff                          : 0.7706
    own_goals                          : 0.6126
    opp_win_rate                       : 0.3807
    team_win_rate                      : 0.3777
    events_team                        : 0.3094
    opponent_position                  : 0.2785
    team_gf                            : 0.2735
    opp_gf                             : 0.2636
    value_diff                         : 0.2606
    player_qual_diff                   : 0.2468
    total_val                          : 0.1471
    avg_val                            : 0.1460
    is_home                            : 0.1208
    total_goals                        : 0.1005


## visualization pipeline
generating visual analytics for interpretation and reporting.

In [22]:
# correlation heatmap
top_20 = list(corr_with_target[corr_with_target.index != 'is_win'].head(18).index) + ['is_win']
corr_mat = df[top_20].corr()

# for all datasets
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('feature correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_01_correlations.png', dpi=300, bbox_inches='tight')
plt.close()

# distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, col in enumerate(['team_win_rate', 'strength_diff', 'value_diff', 'player_qual_diff']):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[df['is_win']==1][col], bins=25, alpha=0.6, label='win', color='#2ecc71')
    ax.hist(df[df['is_win']==0][col], bins=25, alpha=0.6, label='loss', color='#e74c3c')
    ax.set_xlabel(col)
    ax.set_title(f'{col} distribution')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_02_distributions.png', dpi=300, bbox_inches='tight')
plt.close()

## model preparation
preparing features, scaling data, and splitting train/test sets.

In [23]:
exclude_cols = ['game_id', 'club_id', 'opponent_id', 'date', 'hosting', 'is_win', 'type',
                'own_goals', 'opponent_goals', 'competition_id', 'season', 'goal_diff', 'total_goals']

X_cols = [c for c in df.columns if c not in exclude_cols and c in numeric_cols]

X = df[X_cols].values
y = df['is_win'].astype(int).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"  features: {len(X_cols)}")
print(f"  train: {X_train.shape}, test: {X_test.shape}")


  features: 22
  train: (141233, 22), test: (35309, 22)


## model comparison and training
training multiple ensemble learning models and comparing performance.

In [24]:
models = {
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=7, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=7, random_state=42, verbosity=0),
}

results = []
best_model = None
best_name = None
best_auc = 0

for name, model in models.items():
    model.fit(X_train, y_train)

    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(rec, prec)

    results.append({
        'Model': name,
        'Train': train_acc,
        'Test': test_acc,
        'ROC_AUC': roc_auc,
        'PR_AUC': pr_auc,
    })

    if roc_auc > best_auc:
        best_auc = roc_auc
        best_model = model
        best_name = name

comp = pd.DataFrame(results).sort_values('ROC_AUC', ascending=False)
comp.to_csv(r'd:\Projects\WC26Analytics\model_comparison.csv', index=False)

print("\n  Model Results:")
print(comp.to_string(index=False))

# visualization
fig, ax = plt.subplots(figsize=(12, 6))
comp_sort = comp.sort_values('ROC_AUC')
colors = ['#2ecc71' if m == best_name else '#3498db' for m in comp_sort['Model']]
ax.barh(comp_sort['Model'], comp_sort['ROC_AUC'], color=colors)
ax.set_xlabel('ROC AUC')
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.set_xlim([0.5, 1])
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_03_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close()


  Model Results:
            Model    Train     Test  ROC_AUC   PR_AUC
Gradient Boosting 0.855749 0.815939 0.896039 0.855055
          XGBoost 0.889700 0.809029 0.889760 0.847292
    Random Forest 0.915636 0.807500 0.885912 0.841467


## final evaluation and interpretation
evaluating the best-performing model using classification metrics and visualizations.

In [25]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# scores
test_acc = best_model.score(X_test, y_test)
fpr, tpr, _ = roc_curve(y_test, y_proba)
final_roc = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(y_test, y_proba)
final_pr = auc(rec, prec)

# ROC
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='#3498db', lw=3, label=f'ROC (AUC={final_roc:.4f})')
ax.plot([0, 1], [0, 1], 'r--', lw=2, label='Random')
ax.set_xlabel('false positive rate')
ax.set_ylabel('true positive rate')
ax.set_title('ROC curve', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_04_roc_curve.png', dpi=300, bbox_inches='tight')
plt.close()

# PR
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(rec, prec, color='#2ecc71', lw=3, label=f'PR (AUC={final_pr:.4f})')
baseline = y_test.mean()
ax.axhline(baseline, color='#e74c3c', lw=2, linestyle='--', label=f'Baseline ({baseline:.2%})')
ax.set_xlabel('recall')
ax.set_ylabel('precision')
ax.set_title('Precision-Recall curve', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_05_pr_curve.png', dpi=300, bbox_inches='tight')
plt.close()

# confusion
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, annot_kws={'size': 14})
ax.set_xticklabels(['Loss', 'Win'])
ax.set_yticklabels(['Loss', 'Win'])
ax.set_title(f'confusion matrix (Acc: {test_acc:.4f})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_06_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# feature importance
feat_imp = pd.DataFrame({
    'feature': X_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

top_n = 25
top_imp = feat_imp.head(top_n)

fig, ax = plt.subplots(figsize=(12, 10))
colors_imp = plt.cm.viridis(np.linspace(0, 1, len(top_imp)))
ax.barh(range(len(top_imp)), top_imp['importance'].values, color=colors_imp)
ax.set_yticks(range(len(top_imp)))
ax.set_yticklabels(top_imp['feature'].values, fontsize=9)
ax.set_xlabel('importance')
ax.set_title(f'top {top_n} features', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_07_feature_importances.png', dpi=300, bbox_inches='tight')
plt.close()

feat_imp.to_csv(r'd:\Projects\WC26Analytics\feature_importances.csv', index=False)

print(f"\n  top 20 features:")
for idx, row in feat_imp.head(20).iterrows():
    print(f"    {row['feature']:35s} | {row['importance']:.6f}")

# save models
with open(r'd:\Projects\WC26Analytics\best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open(r'd:\Projects\WC26Analytics\scaler_final.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(r'd:\Projects\WC26Analytics\feature_columns_final.pkl', 'wb') as f:
    pickle.dump(X_cols, f)



  top 20 features:
    opp_win_rate                        | 0.226287
    team_win_rate                       | 0.203627
    events_team                         | 0.176886
    events_opp                          | 0.145688
    own_position                        | 0.042581
    is_home                             | 0.033085
    opponent_position                   | 0.028086
    player_qual_diff                    | 0.027464
    value_diff                          | 0.020912
    team_ga                             | 0.009563
    year                                | 0.009297
    opp_ga                              | 0.009221
    team_gf                             | 0.008324
    transfers_team                      | 0.008244
    opp_gf                              | 0.007661
    transfers_opp                       | 0.007641
    total_val_o                         | 0.006791
    avg_val_o                           | 0.006395
    avg_val                             | 0.006350
    total_v

## project summary
final overview of datasets, features, performance metrics, and generated outputs.

In [26]:
print("Project Summary")
print("="*80)
print(f"\nData Summary:")
print(f"  Matches: {len(df)}")
print(f"  Range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"  Features: {len(X_cols)}")
print(f"\nBest Model: {best_name}")
print(f"  Train Acc: {comp.iloc[0]['Train']:.4f}")
print(f"  Test Acc: {comp.iloc[0]['Test']:.4f}")
print(f"  ROC AUC: {final_roc:.4f}")
print(f"  PR AUC: {final_pr:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Loss', 'Win']))

Project Summary

Data Summary:
  Matches: 176542
  Range: 2006-06-09 to 2026-05-07
  Features: 22

Best Model: Gradient Boosting
  Train Acc: 0.8557
  Test Acc: 0.8159
  ROC AUC: 0.8960
  PR AUC: 0.8551

Classification Report:
              precision    recall  f1-score   support

        Loss       0.84      0.87      0.85     21482
         Win       0.78      0.74      0.76     13827

    accuracy                           0.82     35309
   macro avg       0.81      0.80      0.81     35309
weighted avg       0.81      0.82      0.82     35309

